### This notebook compute "VCA18. Index of volume of reservoir storage" indicator for the 27 basins of IKI Project

Spanish: Indice del volumen embalsado

**Created:** 1/9/2026 by Sophia Bakar (sbakar@rti.org)

**Project #:** 0219481  

**Last modified:** 1/12/2026 by Sophia

**Status:** Complete and loaded in SQLite for the baseline scenario.

**QA Status:** reviewed by  

**Original Script Stored at:** Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\Vulnerabilidad

**Packages:** pandas, numpy, geopandas, sqlite3
 
**Inputs:**  The necessary input data are queried from the waterALLOC output database.
 
**Assumptions:** COMIDs that do not have an associated reservoir are assigned a value of NaN.

**Future work:** We should think about an approach for 1. When a reservoir falls within more than 1 COMID, do we consider them equal (so the same values assigned to the multiple COMIDs) or should we weight them based on how much of the reservoir lies within the COMID? 2. If a COMID has more than one reservoir, do we just consider the total average storage between the 2? How will we set a min/max and normalize the values?
 
**Notes:** Run again when future scenario has been run with reservoirs added.

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import sqlite3
import matplotlib.pyplot as plt
import re 

In [2]:
# set up user and database path
#user = 'jmayo'
#user= 'cpickering'
#user = 'sgilson'
#user = 'nreynolds'
user = 'sbakar'
#db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"
# wateralloc_db = fr"C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"
wateralloc_db = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"

In [3]:
# set up indicator ID and get scenarios from database
IndID= 518 #Indicator ID (Exposure = 2 + 0X where X is the Exposure Indicator number, Peligro= 1 +0x, VSB= 3 +0x, VSS= 4 +0x, VCA= 5 +0x)
conn = sqlite3.connect(db_path)

scenarios_df = pd.read_sql_query(
    """
    SELECT ScnID, WaScnName
    FROM ScnMod
    """,
    conn
)

conn.close()

# For now: only baseline and first future
#scenario_ids = scenarios_df.loc[
#    scenarios_df['ScnID'].isin([1, 2]), 'ScnID'
#].tolist()

# for all scenarios:
scenario_ids = scenarios_df['ScnID'].tolist()


In [4]:
scenarios_df

,ScnID,WaScnName
0,1,Linea_Base_2020
1,1,Linea_Base_2020_Embalses
2,2,CC_CMIP6_85_2050


In [5]:
## check what scenarios are available in the WaterALLOC database
# Connect to WaterALLOC database
conn_wa = sqlite3.connect(wateralloc_db)

# Query available scenarios
scenarios_query = """
SELECT DISTINCT Scenario
FROM Scenarios
ORDER BY Scenario
"""

wa_scenarios_df = pd.read_sql_query(scenarios_query, conn_wa)

print("Available scenarios in WaterALLOC DB:")
for s in wa_scenarios_df["Scenario"]:
    print(f" - {s}")



Available scenarios in WaterALLOC DB:
 - CC_CMIP6_85_2050
 - Linea_Base_2020
 - Linea_Base_2020_Embalses


In [ ]:
# define filepaths for input data
#subbasins_shapefile = fr"C:/Users/{user}/Research Triangle Institute/IKI Peru Project - General/Interno/AI2b_Modelacion/Grupos_Modelacion/GIS_WaterALLOC_General/Peru_AHD_with_districts.shp"
subbasins_shapefile = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\GIS_WaterALLOC_General\Peru_AHD_with_districts.shp"


In [ ]:
# read in input data
subbasins_gdf = gpd.read_file(subbasins_shapefile).to_crs('EPSG:32718')

base_comids = subbasins_gdf[['COMID']].drop_duplicates()

In [7]:
# Get monthly average reservoir storage from WaterALLOC database
# (Only for scenarios that include reservoirs: "Embalses")

reservoirstorage_all = []

# Filter scenarios to only those with Embalses
embalses_scenarios = scenarios_df[
    scenarios_df['WaScnName'].str.contains('Embalses', case=False, na=False)
]

# Connect to WaterALLOC database
conn_wa = sqlite3.connect(wateralloc_db)

for scn_id_dynamic, scenario_name in embalses_scenarios.itertuples(index=False):
    print(f"\nProcessing reservoir storage: {scenario_name} (ScnID={scn_id_dynamic})")

    query_storage = """
    SELECT
        a.COMID AS COMID,
        AVG(a.Almacenamiento) AS Storage_Medio,
        COUNT(DISTINCT a.Month) AS n_months
    FROM "WAMSS_Volumen promedio en embalses por COMID" AS a
    JOIN WAMMS_RunsInfo AS b
        ON a.RunID = b.RunID
    JOIN Scenarios AS c
        ON c.ScnID = b.ScnID
    WHERE c.Scenario = ?
    GROUP BY a.COMID
    """

    storage_df = pd.read_sql_query(
        query_storage,
        conn_wa,
        params=(scenario_name,)
    )

    # Add dynamic scenario ID
    storage_df['ScnID_dynamic'] = scn_id_dynamic

    reservoirstorage_all.append(storage_df)

# Close DB connection
conn_wa.close()

# Combine all scenarios into a single DataFrame
reservoirstorage_df = pd.concat(reservoirstorage_all, ignore_index=True)

# Quick summary check
print("\n=== Reservoir storage summary by scenario ===")
print(
    reservoirstorage_df
    .groupby('ScnID_dynamic')['Storage_Medio']
    .agg(['count', 'min', 'max'])
)

print("\nSample rows:")
print(reservoirstorage_df.head())


Processing reservoir storage: Linea_Base_2020_Embalses (ScnID=1)

=== Reservoir storage summary by scenario ===
               count           min          max
ScnID_dynamic                                  
1                  2  36006.060504  69338.46261

Sample rows:
       COMID  Storage_Medio  n_months  ScnID_dynamic
0  310702900   69338.462610        12              1
1  310786600   36006.060504        12              1


In [8]:
reservoirstorage_df['COMID'] = reservoirstorage_df['COMID'].astype(int)
base_comids['COMID'] = base_comids['COMID'].astype(int)

base_index = base_comids.merge(embalses_scenarios[['ScnID']].rename(columns={'ScnID': 'ScnID_dynamic'}), how='cross')

# Merge storage results onto full grid (missing = NaN)
reservoirstorage_df = base_index.merge(reservoirstorage_df,on=['COMID', 'ScnID_dynamic'],how='left')

In [9]:
# Connect to SQLite database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Delete previous entries for this indicator
delete_query = "DELETE FROM IndValues_Dyn WHERE IndID = ?;"
cursor.execute(delete_query, (IndID,))

rows_to_insert = []
for _, row in reservoirstorage_df.iterrows():
    rows_to_insert.append((row['ScnID_dynamic'], IndID, row['COMID'], row['Storage_Medio']))

# Insert data into IndValues_Dyn
insert_query = """
INSERT OR REPLACE INTO IndValues_Dyn (ScnID, IndID, COMID, Value)
VALUES (?, ?, ?, ?);
"""

In [ ]:
# Check that min and max values match the expected range based on the Indicators Table

indicator_limits = pd.read_sql_query(
    """
    SELECT IndID, Min, Max
    FROM Indicators
    WHERE IndID = ?
    """,
    conn,
    params=(IndID,)
)

if indicator_limits.empty:
    raise ValueError(f"No entry found in Indicators table for IndID = {IndID}")

ind_min = indicator_limits.loc[0, 'Min']
ind_max = indicator_limits.loc[0, 'Max']

print(f"\nIndicator {IndID} limits from Indicators table -> Min: {ind_min}, Max: {ind_max}")

# Compute value stats by scenario
value_stats = reservoirstorage_df.groupby('ScnID_dynamic')['Storage_Medio'].agg(['min', 'max', 'count']).reset_index()
print("\n=== Values to be inserted (by scenario) ===")
print(value_stats)

# Check for duplicates in the rows to insert
df_check = pd.DataFrame(rows_to_insert, columns=['ScnID', 'IndID', 'COMID', 'Value'])
duplicates = df_check.duplicated(subset=['ScnID', 'IndID', 'COMID'])
print("\nDuplicates in rows_to_insert:")
print(df_check[duplicates])

In [10]:
#Execute insert to SQLite Database
cursor.executemany(insert_query, rows_to_insert)
conn.commit()
conn.close()